# WalletStory — Reproduce the Theo Case

Runs the full statistical analysis of the Polymarket "Theo" insider trading cluster end-to-end in your browser, using cached on-chain data from the public WalletStory repository.

**No API keys. No local setup. Just Runtime → Run all.**

- [GitHub repo](https://github.com/EmilySun621/Wallet-Story)
- [Live demo](https://wallet-story.vercel.app)
- [Academic foundation: arXiv:2512.18918](https://arxiv.org/html/2512.18918v1)

In [ ]:
!pip install scipy matplotlib requests -q

import json, requests
import numpy as np
from scipy.stats import binomtest, kstest, norm
import matplotlib.pyplot as plt

print("Dependencies loaded.")

## Load Theo case data from GitHub

Pulls case metadata and cached trade history directly from the public repo — no authentication required.

In [ ]:
REPO_RAW = "https://raw.githubusercontent.com/EmilySun621/Wallet-Story/master"

case = requests.get(f"{REPO_RAW}/backend/data/case_polymarket_theo.json").json()
cached = requests.get(f"{REPO_RAW}/backend/data/theo_cluster_trades_cached.json").json()

print(f"Case: {case['case_name']}")
print(f"Seed addresses: {len(case['addresses'])}")
for a in case['addresses']:
    print(f"  • {a['address']}  ({a['username']})")
print(f"\nFull cluster: {len(cached['all_cluster_addresses'])} wallets")
print(f"Total cached trades: {sum(len(t) for t in cached['trades'].values())}")

## Signal 1: Win Rate Anomaly (Binomial test)

Under the null hypothesis of random 50% win rate, what's the probability of observing this many wins across the cluster?

In [ ]:
all_trades = [t for trades in cached['trades'].values() for t in trades]
wins = sum(1 for t in all_trades if t.get('won') is True)
losses = sum(1 for t in all_trades if t.get('won') is False)
total = wins + losses

print(f"Directional trades: {total}")
print(f"Wins: {wins} ({wins/total*100:.2f}%)")
print(f"Losses: {losses}")

result = binomtest(wins, total, p=0.5, alternative='greater')
p_val = result.pvalue
print(f"\nBinomial p-value: {p_val:.3e}")
print("Verdict:", "CRITICAL" if p_val < 1e-10 else "HIGH" if p_val < 1e-5 else "MEDIUM" if p_val < 0.01 else "LOW")

## Signal 4a: Timing Distribution Anomaly (KS test)

Test whether the cluster's aggregate trade timing is uniformly distributed. Uniform = baseline. Non-uniform = coordinated patterns.

In [ ]:
all_ts = sorted([t['timestamp'] for trades in cached['trades'].values() for t in trades])
t_min, t_max = all_ts[0], all_ts[-1]
normalized = [(t - t_min) / (t_max - t_min) for t in all_ts]

ks_stat, ks_p = kstest(normalized, 'uniform')
print(f"KS statistic: {ks_stat:.4f}")
print(f"KS p-value: {ks_p:.3e}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(normalized, bins=20, color='#8B5CF6', alpha=0.85, edgecolor='white')
ax.axhline(len(normalized)/20, color='#10B981', linestyle='--', label='Uniform baseline')
ax.set_xlabel('Normalized trade timestamp')
ax.set_ylabel('Count')
ax.set_title(f'Theo cluster timing — KS p = {ks_p:.2e}')
ax.legend()
plt.tight_layout()
plt.show()

## ⭐ Signal 4b: Pairwise Temporal Alignment

Extends Signal 4 with cluster-level coordination check from [arXiv:2512.18918](https://arxiv.org/html/2512.18918v1).

For every pair (i, j), score how closely their trade timestamps align. Kernel: same hour = 1.0, 24h+ apart = 0.0.

Synchronized trading across multiple wallets is a stronger signal than any individual timing pattern.

In [ ]:
def alignment_score(ta, tb, kernel_sec=86400):
    if not ta or not tb:
        return 0.0
    tsa = sorted(t['timestamp'] for t in ta)
    tsb = sorted(t['timestamp'] for t in tb)
    return np.mean([max(0, 1 - abs(t - min(tsb, key=lambda x: abs(x-t)))/kernel_sec) for t in tsa])

addresses = list(cached['trades'].keys())
n = len(addresses)
M = np.ones((n, n))

for i in range(n):
    for j in range(i+1, n):
        s = alignment_score(cached['trades'][addresses[i]], cached['trades'][addresses[j]])
        M[i][j] = M[j][i] = s

off_diag = M[~np.eye(n, dtype=bool)]
observed = off_diag.mean()
print(f"Observed mean alignment: {observed:.3f}")

# Null model: shuffle timestamps within each wallet, recompute
np.random.seed(42)
null_means = []
for _ in range(20):
    shuffled = {}
    for addr, trades in cached['trades'].items():
        ts = [t['timestamp'] for t in trades]
        if not ts:
            shuffled[addr] = []
            continue
        new_ts = np.random.uniform(min(ts), max(ts), len(trades))
        shuffled[addr] = [{ **t, 'timestamp': nt} for t, nt in zip(trades, new_ts)]
    sim = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            sim[i][j] = sim[j][i] = alignment_score(shuffled[addresses[i]], shuffled[addresses[j]])
    null_means.append(sim[~np.eye(n, dtype=bool)].mean())

null_mean = np.mean(null_means)
null_std = np.std(null_means)
z = (observed - null_mean) / null_std if null_std > 0 else 0
p_align = 1 - norm.cdf(z)

print(f"Null mean: {null_mean:.3f} ± {null_std:.3f}")
print(f"Z-score: {z:.2f}σ")
print(f"One-tailed p-value: {p_align:.3e}")

In [ ]:
# Heatmap — the visual payoff
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(M, cmap='plasma', vmin=0, vmax=1)
short = [a[:8]+'…' for a in addresses]
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(short, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(short, fontsize=8)
ax.set_title(f'Pairwise Temporal Alignment — Theo Cluster ({z:.1f}σ above null)')
plt.colorbar(im, ax=ax, label='Alignment score')
plt.tight_layout()
plt.show()

print(f"\nHigh off-diagonal values = cluster members trade in synchronized patterns.")
print(f"Under null hypothesis (uncoordinated trading), p = {p_align:.3e}.")

## Summary

| Signal | Result |
|---|---|
| Win Rate Anomaly (binomial) | p ≈ 0 (below float64 precision) |
| Timing KS test | See above |
| Pairwise Temporal Alignment | Z-score displayed in cell above |
| Cluster size | 13 wallets |

All reproduced locally in your browser. Data source: public GitHub repo. Zero API keys.

**Next steps:**
- Try the [live demo](https://wallet-story.vercel.app) — paste Theo4 seed `0x56687bf447db6ffa42ffe2204a05edaa20f55839`
- Read the [full methodology](https://wallet-story.vercel.app/methodology)
- View on-chain EAS attestations on [Sepolia EASScan](https://sepolia.easscan.org)